# K3A × Sentinel-2 전처리 파이프라인 (v9-A)

## 처리 내용
1. 공통 설정
2. CDSE 로그인 및 API 함수
3. K3A 씬 목록 구성
4. Sentinel-2 L1C 다운로드
5. K3A 정사보정 (RPC + GLO-30 DEM)
6. 결과 저장 (ortho_result_map.json)

**출력**: `k3a_ortho/` 폴더 + `ortho_result_map.json`
→ 노트북 B (칩 생성)에서 이어서 사용


## 0. 설치 및 공통 설정

In [ ]:
# ======================================================================
# 경로 설정 — 본인 환경에 맞게 수정하세요
# ======================================================================
import os

# Colab을 쓰는 경우에만 drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_ROOT = '/content/drive/MyDrive/kompsat_sr'  # ← 본인 드라이브 경로로 수정
except ImportError:
    DEFAULT_ROOT = './data'  # 로컬 환경

PROJECT_ROOT     = os.environ.get('PROJECT_ROOT', DEFAULT_ROOT)
DATASET_ROOT     = os.path.join(PROJECT_ROOT, 'dataset')
CHECKPOINT_DIR   = os.path.join(PROJECT_ROOT, 'checkpoints')
PRITHVI_DIR      = os.path.join(PROJECT_ROOT, 'prithvi')
SR_ROOT          = PROJECT_ROOT
VAL_DATASET_ROOT = os.path.join(PROJECT_ROOT, 'val_dataset')

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'✅ PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
!pip install -q rasterio shapely tqdm
!apt-get install -q -y gdal-bin python3-gdal

import shutil
import os, glob, zipfile, subprocess, math, warnings, re, json
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import Affine
import requests, getpass
from datetime import datetime, timedelta
from shapely.geometry import box, shape
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

BASE_DIR     = f'{PROJECT_ROOT}'
DOWNLOAD_DIR = os.path.join(BASE_DIR, 'download_data')
EXTRACT_DIR  = os.path.join(BASE_DIR, 'extracted')
DEM_DIR      = os.path.join(BASE_DIR, 'dem')
ORTHO_DIR    = os.path.join(BASE_DIR, 'k3a_ortho')

# 정사보정 결과 맵 저장 경로 (노트북 B에서 로드)
ORTHO_MAP_JSON = os.path.join(BASE_DIR, 'ortho_result_map.json')

S2_BANDS  = {'R': 'B04', 'G': 'B03', 'B': 'B02', 'NIR': 'B08'}
K3A_BANDS = {'R': '_R.tif', 'G': '_G.tif', 'B': '_B.tif', 'NIR': '_N.tif'}

for d in [DOWNLOAD_DIR, EXTRACT_DIR, DEM_DIR, ORTHO_DIR]:
    os.makedirs(d, exist_ok=True)

print('환경 설정 완료')


## 1. CDSE 로그인 및 API 함수

In [ ]:
print('🌐 CDSE 로그인')
CDSE_USERNAME = input('CDSE 이메일: ')
CDSE_PASSWORD = getpass.getpass('비밀번호: ')

def get_cdse_token(username, password):
    resp = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={'client_id': 'cdse-public', 'grant_type': 'password',
              'username': username, 'password': password})
    resp.raise_for_status()
    return resp.json()['access_token']

def get_footprint_from_rpc(rpc_path):
    rpc = {}
    with open(rpc_path) as f:
        for line in f:
            if ':' in line:
                k, v = line.split(':', 1)
                try: rpc[k.strip()] = float(v.strip().split()[0])
                except ValueError: pass
    lat_off, lat_sc = rpc.get('LAT_OFF', 0), rpc.get('LAT_SCALE', 0)
    lon_off, lon_sc = rpc.get('LONG_OFF', 0), rpc.get('LONG_SCALE', 0)
    return [lon_off-lon_sc, lat_off-lat_sc, lon_off+lon_sc, lat_off+lat_sc]

def search_s2_stac(bbox, start_date, end_date, cloud_pct=10, limit=1):
    resp = requests.post(
        'https://catalogue.dataspace.copernicus.eu/stac/search',
        json={'collections': ['sentinel-2-l1c'],
              'datetime': f'{start_date}T00:00:00Z/{end_date}T23:59:59Z',
              'bbox': bbox,
              'query': {'eo:cloud_cover': {'lte': cloud_pct}},
              'sortby': [{'field': 'eo:cloud_cover', 'direction': 'asc'}],
              'limit': limit})
    resp.raise_for_status()
    return resp.json()

def get_uuid_from_title(title):
    resp = requests.get(
        f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{title}'")
    if resp.status_code == 200:
        vals = resp.json().get('value', [])
        if vals: return vals[0]['Id']
    return None

def download_product(product_uuid, token, save_path):
    resp = requests.get(
        f'https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_uuid})/$value',
        headers={'Authorization': f'Bearer {token}'}, stream=True)
    if resp.status_code == 202: raise RuntimeError('오프라인 상태')
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    with open(save_path, 'wb') as f, tqdm(
            desc=os.path.basename(save_path), total=total,
            unit='iB', unit_scale=True, unit_divisor=1024) as bar:
        for chunk in resp.iter_content(8192):
            f.write(chunk); bar.update(len(chunk))

print('✅ API 함수 정의 완료')


## 2. K3A 씬 목록 구성

In [ ]:
region_folders = sorted(
    f for f in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, f)) and f[0].isdigit())

all_k3a_list = [
    (region, k3a)
    for region in region_folders
    for k3a in sorted(os.listdir(os.path.join(BASE_DIR, region)))
    if k3a.startswith('K3A')]

print(f'지역 폴더: {region_folders}')
print(f'K3A 씬 수: {len(all_k3a_list)}개')


## 3. Sentinel-2 L1C 다운로드

### 3-1. 1차 탐색 (±10일)

In [ ]:
bad_matches_k3a = []

try:
    token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)
    for region in region_folders:
        region_path = os.path.join(BASE_DIR, region)
        k3a_folders = sorted(f for f in os.listdir(region_path)
            if os.path.isdir(os.path.join(region_path, f)) and f.startswith('K3A'))
        print(f'\n🗺️  지역: {region}  ({len(k3a_folders)}개)')
        for k3a_folder in k3a_folders:
            k3a_path  = os.path.join(region_path, k3a_folder)
            rpc_files = glob.glob(os.path.join(k3a_path, '*_rpc.txt'))
            if not rpc_files: print(f'  ⚠️  {k3a_folder}: RPC 없음'); continue
            bbox     = get_footprint_from_rpc(rpc_files[0])
            k3a_poly = box(*bbox)
            k3a_date = datetime.strptime(k3a_folder.split('_')[1][:8], '%Y%m%d')
            s_date   = (k3a_date - timedelta(days=10)).strftime('%Y-%m-%d')
            e_date   = (k3a_date + timedelta(days=10)).strftime('%Y-%m-%d')
            results  = search_s2_stac(bbox, s_date, e_date, cloud_pct=10, limit=5)
            matched  = False
            for feat in results.get('features', []):
                s2_geom = shape(feat['geometry'])
                if (k3a_poly.intersection(s2_geom).area / k3a_poly.area) < 0.999: continue
                s2_title = feat.get('properties', {}).get('title', feat.get('id', ''))
                if not s2_title.endswith('.SAFE'): s2_title += '.SAFE'
                save_path = os.path.join(DOWNLOAD_DIR,
                    f"{k3a_folder}_{s2_title.replace('.SAFE','')}.zip")
                if os.path.exists(save_path):
                    print(f'  ℹ️  {k3a_folder}: 이미 존재'); matched = True; break
                s2_uuid = get_uuid_from_title(s2_title)
                if not s2_uuid: continue
                print(f'  📥 {k3a_folder} ← {s2_title[:55]}...')
                try:
                    token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)
                    download_product(s2_uuid, token, save_path)
                    matched = True; break
                except Exception as e:
                    print(f'  ⚠️  다운로드 실패: {e}')
                    if os.path.exists(save_path): os.remove(save_path)
            if not matched:
                print(f'  ❌ {k3a_folder}: 1차 탐색 실패')
                bad_matches_k3a.append(k3a_folder)
except Exception as e:
    print(f'❌ 오류: {e}')
print(f'\n재탐색 대기열: {len(bad_matches_k3a)}건')


### 3-2. 2차 탐색 (±30일)

In [ ]:
print(f'🔄 재탐색 시작 ({len(bad_matches_k3a)}건)')
for k3a_folder in bad_matches_k3a:
    if glob.glob(os.path.join(DOWNLOAD_DIR, f'{k3a_folder}_*.zip')):
        print(f'  ℹ️  {k3a_folder}: 이미 존재'); continue
    rpc_file = glob.glob(os.path.join(BASE_DIR, '*', k3a_folder, '*_rpc.txt'))
    if not rpc_file: print(f'  ⚠️  {k3a_folder}: RPC 없음'); continue
    bbox     = get_footprint_from_rpc(rpc_file[0])
    k3a_poly = box(*bbox)
    k3a_date = datetime.strptime(k3a_folder.split('_')[1][:8], '%Y%m%d')
    s_date   = (k3a_date - timedelta(days=30)).strftime('%Y-%m-%d')
    e_date   = (k3a_date + timedelta(days=30)).strftime('%Y-%m-%d')
    results  = search_s2_stac(bbox, s_date, e_date, cloud_pct=10, limit=20)
    for old in glob.glob(os.path.join(DOWNLOAD_DIR, f'{k3a_folder}_*.zip')):
        os.remove(old)
    found = False
    for feat in results.get('features', []):
        s2_geom = shape(feat['geometry'])
        if (k3a_poly.intersection(s2_geom).area / k3a_poly.area) < 0.999: continue
        s2_title = feat.get('properties', {}).get('title', feat.get('id', ''))
        if not s2_title.endswith('.SAFE'): s2_title += '.SAFE'
        save_path = os.path.join(DOWNLOAD_DIR,
            f"{k3a_folder}_{s2_title.replace('.SAFE','')}.zip")
        s2_uuid = get_uuid_from_title(s2_title)
        if not s2_uuid: continue
        print(f'  ✅ {k3a_folder} ← {s2_title[:55]}...')
        try:
            token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)
            download_product(s2_uuid, token, save_path)
            found = True; break
        except Exception as e:
            print(f'  ⚠️  실패: {e}')
            if os.path.exists(save_path): os.remove(save_path)
    if not found: print(f'  ❌ {k3a_folder}: ±30일에도 실패')


### 3-3. 다운로드 결과 요약

In [ ]:
all_k3a = [k3a for region, k3a in all_k3a_list]
downloaded_zips = glob.glob(os.path.join(DOWNLOAD_DIR, '*.zip'))
successful_k3a = []
for z in downloaded_zips:
    name = os.path.basename(z)
    for sat in ('_S2A_', '_S2B_'):
        if sat in name: successful_k3a.append(name.split(sat)[0]); break
failed_k3a = sorted(set(all_k3a) - set(successful_k3a))
print(f'총 K3A: {len(all_k3a)}개  ✅ 성공: {len(successful_k3a)}개  ❌ 실패: {len(failed_k3a)}개')
if failed_k3a:
    for f in failed_k3a: print(f'  - {f}')


## 4. K3A 정사보정 (RPC + GLO-30 DEM)

### 4-1. 정사보정 유틸리티 함수

In [ ]:
def _tile_name(lat, lon):
    lat_h = 'N' if lat >= 0 else 'S'
    lon_h = 'E' if lon >= 0 else 'W'
    return (f'Copernicus_DSM_COG_10_{lat_h}{abs(lat):02d}_00'
            f'_{lon_h}{abs(lon):03d}_00_DEM')

def download_glo30_tile(lat, lon, tile_dir):
    name     = _tile_name(lat, lon)
    out_path = os.path.join(tile_dir, f'{name}.tif')
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        return out_path
    url  = f'https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif'
    resp = requests.get(url, stream=True, timeout=120)
    if resp.status_code in (403, 404): return None
    resp.raise_for_status()
    with open(out_path, 'wb') as f:
        for chunk in resp.iter_content(65536): f.write(chunk)
    return out_path

def build_scene_dem(k3a_folder, bbox_padded, dem_dir):
    scene_dem = os.path.join(dem_dir, f'{k3a_folder}_dem.tif')
    if os.path.exists(scene_dem) and os.path.getsize(scene_dem) > 10_000:
        return scene_dem
    tile_dir = os.path.join(dem_dir, 'tiles')
    os.makedirs(tile_dir, exist_ok=True)
    lon_min, lat_min, lon_max, lat_max = bbox_padded
    tiles = [(lat, lon)
             for lat in range(int(math.floor(lat_min)), int(math.ceil(lat_max)))
             for lon in range(int(math.floor(lon_min)), int(math.ceil(lon_max)))]
    tile_paths = [p for lat, lon in tiles
                  for p in [download_glo30_tile(lat, lon, tile_dir)] if p]
    if not tile_paths: raise RuntimeError(f'GLO-30 타일 없음: {bbox_padded}')
    cmd = ['gdal_merge.py', '-o', scene_dem,
           '-ul_lr', str(lon_min), str(lat_max), str(lon_max), str(lat_min),
           '-co', 'COMPRESS=LZW', '-co', 'TILED=YES', '-a_nodata', '-9999'
           ] + tile_paths
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f'gdal_merge 실패:\n{r.stderr}')
    return scene_dem

def parse_rpc_txt(rpc_path):
    """
    K3A RPC txt 파일 파싱.
    COEFF 20개 행을 하나의 공백 구분 문자열로 합쳐서 반환.
    """
    scalar_keys = {
        'LINE_OFF', 'SAMP_OFF', 'LAT_OFF', 'LONG_OFF', 'HEIGHT_OFF',
        'LINE_SCALE', 'SAMP_SCALE', 'LAT_SCALE', 'LONG_SCALE', 'HEIGHT_SCALE',
    }
    coeff_groups = {
        'LINE_NUM_COEFF': {},
        'LINE_DEN_COEFF': {},
        'SAMP_NUM_COEFF': {},
        'SAMP_DEN_COEFF': {},
    }

    rpc = {}

    with open(rpc_path) as f:
        for line in f:
            line = line.strip()
            if ':' not in line:
                continue
            key, val = line.split(':', 1)
            key = key.strip()
            val = val.strip().split()[0].lstrip('+')  # 단위 제거, + 제거

            # 스칼라 키 (OFF, SCALE)
            if key in scalar_keys:
                try:
                    rpc[key] = val
                except:
                    pass
                continue

            # COEFF 키 (LINE_NUM_COEFF_1 ~ _20 등)
            matched = False
            for group in coeff_groups:
                if key.startswith(group + '_'):
                    idx_str = key[len(group)+1:]  # '_1' → '1'
                    try:
                        idx = int(idx_str)
                        coeff_groups[group][idx] = val
                        matched = True
                    except:
                        pass
                    break

    # COEFF 그룹을 순서대로 공백 구분 문자열로 합치기
    for group, vals in coeff_groups.items():
        if vals:
            ordered = [vals[i] for i in sorted(vals.keys())]
            rpc[group] = ' '.join(ordered)

    return rpc

def orthorectify_k3a_band(input_tif, rpc_txt, dem_path, output_tif,
                           target_res=2.5, target_epsg=32652):
    if os.path.exists(output_tif):
        return output_tif
    os.makedirs(os.path.dirname(output_tif) or '.', exist_ok=True)

    tmp_tif = output_tif.replace('.tif', '_rpc_tmp.tif')
    rpc     = parse_rpc_txt(rpc_txt)

    # 원본 복사 후 RPC 도메인에 직접 주입 + GCP 제거
    shutil.copy(input_tif, tmp_tif)
    ds = gdal.Open(tmp_tif, gdal.GA_Update)
    ds.SetMetadata(rpc, 'RPC')
    ds.SetGCPs([], '')
    ds.FlushCache()
    ds = None

    cmd_warp = ['gdalwarp', '-rpc',
                '-to', f'RPC_DEM={dem_path}',
                '-t_srs', f'EPSG:{target_epsg}',
                '-tr', str(target_res), str(target_res),
                '-r', 'cubic',
                '-co', 'COMPRESS=LZW', '-co', 'TILED=YES',
                '-wo', 'NUM_THREADS=ALL_CPUS',
                tmp_tif, output_tif]
    r = subprocess.run(cmd_warp, capture_output=True, text=True)
    if os.path.exists(tmp_tif):
        os.remove(tmp_tif)
    if r.returncode != 0:
        raise RuntimeError(f'gdalwarp 실패:\n{r.stderr}')
    return output_tif

def orthorectify_k3a_scene(k3a_folder, k3a_path, ortho_dir, dem_path):
    out_stack = os.path.join(ortho_dir, f'{k3a_folder}_ortho.tif')
    if os.path.exists(out_stack): return out_stack
    rpc_files = (glob.glob(os.path.join(k3a_path, '*_M_rpc.txt')) or
                 glob.glob(os.path.join(k3a_path, '*_rpc.txt')))
    if not rpc_files: raise FileNotFoundError(f'RPC 없음: {k3a_path}')
    rpc_txt = rpc_files[0]
    band_orthos = []
    for key, suffix in K3A_BANDS.items():
        tif_list = glob.glob(os.path.join(k3a_path, f'*{suffix}'))
        if not tif_list: raise FileNotFoundError(f'K3A {key} 밴드 없음')
        out_band = os.path.join(ortho_dir, f'{k3a_folder}_{key}_ortho.tif')
        print(f'    {key} 밴드 정사보정...')
        orthorectify_k3a_band(tif_list[0], rpc_txt, dem_path, out_band)
        band_orthos.append(out_band)
    with rasterio.open(band_orthos[0]) as ref:
        meta = ref.meta.copy(); meta.update(count=4, compress='lzw')
    with rasterio.open(out_stack, 'w', **meta) as dst:
        for i, bp in enumerate(band_orthos, start=1):
            with rasterio.open(bp) as src: dst.write(src.read(1), i)
    for bp in band_orthos: os.remove(bp)
    return out_stack

print('✅ 정사보정 유틸리티 함수 정의 완료')


### 4-2. GLO-30 DEM 다운로드

In [ ]:
scene_dem_map = {}

for region, k3a_folder in tqdm(all_k3a_list, desc='GLO-30 DEM'):
    k3a_path  = os.path.join(BASE_DIR, region, k3a_folder)
    rpc_files = glob.glob(os.path.join(k3a_path, '*_rpc.txt'))
    if not rpc_files: print(f'  ⚠️  {k3a_folder}: RPC 없음'); continue
    bbox        = get_footprint_from_rpc(rpc_files[0])
    bbox_padded = [bbox[0]-0.05, bbox[1]-0.05, bbox[2]+0.05, bbox[3]+0.05]
    try:
        dem_path = build_scene_dem(k3a_folder, bbox_padded, DEM_DIR)
        scene_dem_map[k3a_folder] = dem_path
        print(f'  ✅ {k3a_folder}')
    except Exception as e:
        print(f'  ❌ {k3a_folder}: {e}')

print(f'DEM 준비: {len(scene_dem_map)}/{len(all_k3a_list)}개')


### 4-3. K3A 정사보정 실행

In [ ]:
ortho_result_map = {}

for region, k3a_folder in tqdm(all_k3a_list, desc='K3A 정사보정'):
    k3a_path = os.path.join(BASE_DIR, region, k3a_folder)
    if k3a_folder not in scene_dem_map:
        print(f'  ⚠️  {k3a_folder}: DEM 없음'); continue
    print(f'\n🔄 {k3a_folder}')
    try:
        ortho_path = orthorectify_k3a_scene(
            k3a_folder, k3a_path, ORTHO_DIR,
            dem_path=scene_dem_map[k3a_folder])
        ortho_result_map[k3a_folder] = ortho_path
        print(f'  ✅ {os.path.basename(ortho_path)}')
    except Exception as e:
        print(f'  ❌ 오류: {e}')

print(f'정사보정 완료: {len(ortho_result_map)}/{len(all_k3a_list)}개')


## 5. 결과 저장 → 노트북 B에서 로드

In [ ]:
# ortho_result_map을 JSON으로 저장
# 노트북 B (칩 생성)에서 이 파일을 로드해서 이어서 사용
with open(ORTHO_MAP_JSON, 'w') as f:
    json.dump(ortho_result_map, f, ensure_ascii=False, indent=2)

print(f'✅ 저장 완료: {ORTHO_MAP_JSON}')
print(f'   정사보정 씬 수: {len(ortho_result_map)}개')
for k, v in list(ortho_result_map.items())[:3]:
    print(f'   {k}: {os.path.basename(v)}')
if len(ortho_result_map) > 3:
    print(f'   ... 외 {len(ortho_result_map)-3}개')
